# P5 lab: exact vector search in NumPy

Lesson: [`concepts/P5_vector_search.md`](../concepts/P5_vector_search.md), Part H.1. Concept card: [02 P5](../02_RAG_MASTER_LEARNING_PLAN.md#p5).

**How this lab works (blank-file first).**
- Each task has a **task cell** (what to build), an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. You may look up NumPy and sentence-transformers docs. The reference script `sources/verification/P5/p5_measure.py` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- Where a task says **Predict first**, fill the prediction cell **before** you run anything. The CHECK cells refuse to run without it.
- The CHECK cells compare **fingerprints** (`_fp`), so the expected answers stay hidden until you have predicted.
- **No API calls** in this lab (no Groq, no Cohere). The embedding model is the local MiniLM from your Hugging Face cache.
- Kernel: the project uv environment (`rag-learning`). Run the cells top to bottom.

## Setup (given)

Run this cell. It loads the climate PDF the same way as the acceptance suite (pages joined with one `"\n"`), gives you `page_of(offset)`, and defines two helpers the CHECK cells use: `_fp` (a fingerprint) and `_names` (the names a function uses).

In [ ]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # never download; MiniLM is in the local cache
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import hashlib, json, time
import numpy as np
import pymupdf

PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"
MINILM = "sentence-transformers/all-MiniLM-L6-v2"
Q_D002 = "In what year was the Paris Agreement adopted?"     # 06 seed D-002 (draft item; demo only)
Q_D008 = "What does the Kigali Amendment target?"           # 06 seed D-008 (draft item; demo only)

pages = [p.get_text() for p in pymupdf.open(PDF)]
full = "\n".join(pages)
starts = [sum(len(x) + 1 for x in pages[:i]) for i in range(len(pages))]

def page_of(offset):
    """1-based page number of a character offset in `full`."""
    return max(i + 1 for i, s0 in enumerate(starts) if s0 <= offset)

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

def _names(fn):
    """Every global or attribute name a function's code uses (also inside nested code)."""
    out, todo = set(), [fn.__code__]
    while todo:
        c = todo.pop()
        out |= set(c.co_names)
        todo += [k for k in c.co_consts if hasattr(k, "co_names")]
    return out

print(len(pages), "pages,", len(full), "characters; numpy", np.__version__, "; pymupdf", pymupdf.VersionBind)

## Task 1: chunks with metadata stored alongside

Build two lists **of the same length**, where position `i` in one belongs to position `i` in the other:
- `chunks`: plain fixed character windows of `full`, size 1,000, overlap 200 (starts at 0, 800, 1600, ...; the last one may be shorter). Not LangChain: plain slices.
- `meta`: one dict per chunk with the keys `chunk_id` (= its position), `page` (= `page_of(char_start)`), `char_start`, `char_end`.

Why: the vector index will only hold numbers. Row `i` of the index must still lead you back to the text and the page you will cite.

In [ ]:
# Your code here (Task 1). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 1 (do not edit)
assert isinstance(chunks, list) and isinstance(meta, list), "chunks and meta must be lists"
assert len(chunks) == len(meta), "chunks and meta must have the same length (row i belongs to row i)"
assert len(chunks) == 91, f"expected 91 windows of 1000/200 over 72,593 characters, got {len(chunks)}"
assert all(set(m) >= {"chunk_id", "page", "char_start", "char_end"} for m in meta), "each meta dict needs chunk_id, page, char_start, char_end"
assert all(m["chunk_id"] == i for i, m in enumerate(meta)), "chunk_id must equal the position"
assert all(chunks[i] == full[m["char_start"]:m["char_end"]] for i, m in enumerate(meta)), "chunk text must equal full[char_start:char_end]"
assert meta[1]["char_start"] == 800, "the stride is size - overlap = 800"
assert _fp([m["page"] for m in meta]) == "62fa9c745825b2e2", "pages are not right yet: page = page_of(char_start)"
print("Task 1 OK:", len(chunks), "chunks; last one", meta[-1])

## Task 2: embed every chunk once

**Predict first.** MiniLM ends with a `Normalize` module. If you call `encode(..., normalize_embeddings=False)`, what will the length (L2 norm) of each vector be? Write a number.

In [ ]:
my_prediction_norm = None   # replace None with a number, BEFORE you embed

Now build:
- `model`: `SentenceTransformer(MINILM, device="cpu")`;
- `raw_norms`: a NumPy array with the L2 norm of each chunk vector from `encode(chunks, normalize_embeddings=False)`;
- `V`: the chunk vectors from `encode(chunks, normalize_embeddings=True)`, as a `float32` array of shape `(91, 384)`. Row `i` of `V` is chunk `i`.

In [ ]:
# Your code here (Task 2). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_norm is not None, "write your prediction first (it is part of the evidence)"
assert isinstance(V, np.ndarray) and V.shape == (91, 384) and V.dtype == np.float32, f"V must be float32 (91, 384), got {getattr(V, 'shape', None)} {getattr(V, 'dtype', None)}"
assert len(V) == len(meta), "one vector per metadata row"
assert np.allclose(np.linalg.norm(V, axis=1), 1.0, atol=1e-5), "rows of V must have length 1"
assert len(raw_norms) == 91, "raw_norms needs one norm per chunk"
print(f"Task 2 OK. Your prediction {my_prediction_norm}; measured raw norms {raw_norms.min():.6f} .. {raw_norms.max():.6f}")

## Task 3: the reference, written the slow and obvious way

Write `brute_force_top_k(scores, k)`:
- input: a sequence of scores (higher = more similar) and `k`;
- output: a **Python list** of the `k` positions with the highest scores, best first;
- **ties:** when two scores are equal, the **smaller position** comes first;
- use plain Python only (`float(...)`, `sorted`, `range`, a `key=`). No `np.argsort`, no `np.argpartition`.

This is your ground truth for Task 4. It is slow on purpose: it is easy to read and hard to get wrong.

In [ ]:
# Your code here (Task 3). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 3 (do not edit)
_toy = [0.70, 0.90, 0.80, 0.80, 0.80, 0.10]
assert not (_names(brute_force_top_k) & {"argsort", "argpartition", "lexsort"}), "the reference must not use NumPy sorting"
_r = brute_force_top_k(_toy, 2)
assert isinstance(_r, list) and len(_r) == 2, "return a Python list of k positions"
assert _fp([int(i) for i in _r]) == "3a316d6d3226f84c", f"toy k=2 gives {_r}: check the order (highest first) and the tie rule (smaller position first)"
assert _fp([int(i) for i in brute_force_top_k(_toy, 4)]) == "d1be96124d992521", "toy k=4 is not right yet: check the tie rule"
print("Task 3 OK:", _r, brute_force_top_k(_toy, 4))

## Task 4: fast exact top-k with `np.argpartition`

**Predict first.** For the toy scores `[0.70, 0.90, 0.80, 0.80, 0.80, 0.10]` and k = 2, which positions should come back, in order? Three positions tie for second place.

In [ ]:
my_prediction_toy_ids = None   # e.g. [5, 0]; replace None BEFORE you code

Write `top_k(scores, k)` for a 1-D NumPy array:
1. use `np.argpartition` to find the k best **without sorting the whole array**;
2. then order those k by score, highest first;
3. make ties deterministic: equal scores → smaller position first. Careful: at the boundary, `argpartition` may pick **any** of the tied positions, so make sure the smallest tied positions win;
4. return a NumPy integer array of length `min(k, len(scores))`.

The CHECK compares your `top_k` with your `brute_force_top_k` on 300 random score vectors full of ties.

In [ ]:
# Your code here (Task 4). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 4 (do not edit)
assert my_prediction_toy_ids is not None, "write your prediction first"
assert "argpartition" in _names(top_k), "use np.argpartition (the point of this task)"
_t = np.array([0.70, 0.90, 0.80, 0.80, 0.80, 0.10], dtype=np.float32)
assert _fp(top_k(_t, 2).tolist()) == "3a316d6d3226f84c", f"toy k=2 gives {top_k(_t, 2).tolist()}"
_rng = np.random.default_rng(12345)
for _trial in range(300):
    _n = int(_rng.integers(5, 60)); _k = int(_rng.integers(1, 12))
    _s = np.round(_rng.random(_n), 1).astype(np.float32)           # rounding to 1 decimal makes many ties
    _got = [int(i) for i in top_k(_s, _k)]
    _want = brute_force_top_k(_s, _k)
    assert _got == _want, f"trial {_trial}: top_k={_got} but brute force={_want} for k={_k}, scores={_s.tolist()}"
    # independent property check: every left-out position is worse, or equal with a larger position
    _worst = _got[-1]
    for _j in set(range(_n)) - set(_got):
        assert (_s[_j], -_j) < (_s[_worst], -_worst), f"trial {_trial}: position {_j} should have been in the top-{_k}"
print(f"Task 4 OK. Your prediction {my_prediction_toy_ids}; top_k gives {top_k(_t, 2).tolist()}; 300 tie-heavy trials agree with brute force")

## Task 5: one real query (MC-002)

**Predict first.** (a) The query vector `q` has shape `(384,)`. What is the shape of the score array for one query against all 91 chunks? (b) Which page do you expect the best chunk for the Paris Agreement question to come from?

In [ ]:
my_prediction_scores_shape = None   # e.g. (91, 91) or (91,) or (384,)
my_prediction_top_page = None       # a page number

Build:
- `q`: the embedding of `Q_D002`, made with the **same** model and the same normalisation as the chunks, shape `(384,)`;
- `scores`: one score per chunk (cosine, because all vectors have length 1);
- `ids = top_k(scores, 4)`;
- `results`: a list of 4 dicts with the keys `rank` (1-4), `chunk_id`, `page`, `score`, built by looking up `meta`.

Print `results` and read the four chunks' first 100 characters.

In [ ]:
# Your code here (Task 5). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 5 (do not edit)
assert my_prediction_scores_shape is not None and my_prediction_top_page is not None, "write both predictions first"
assert q.shape == (384,), f"q must be one vector of shape (384,), got {q.shape}"
assert abs(float(np.linalg.norm(q)) - 1) < 1e-5, "q must be normalised like the chunks"
assert scores.shape == (91,), f"one query needs ONE score per chunk: shape (91,), got {scores.shape}. A (91, 91) all-pairs matrix is not retrieval (MC-002)."
assert _fp([int(i) for i in ids]) == "2df9fe7b18fb4d83", f"top-4 ids {list(map(int, ids))} are not right yet: same model, normalised, scores = V @ q?"
assert [r["rank"] for r in results] == [1, 2, 3, 4], "ranks 1..4"
assert _fp([int(r["page"]) for r in results]) == "27564e53c9dd6b7c", "the pages in results do not match meta"
print(f"Task 5 OK. Predicted shape {my_prediction_scores_shape}, page {my_prediction_top_page}; got {scores.shape}, page {results[0]['page']}")
for r in results:
    print(r)

## Task 5b: ties in a real index (duplicates)

`dartboard.ipynb` cell 9 loads every document 5 times to imitate a dense, redundant collection. Do the same: `V5 = np.tile(V, (5, 1))`, so row `r` is chunk `r % 91`.

**Predict first.** How many **different** chunks will be in the top-4 of `V5 @ q`?

In [ ]:
my_prediction_distinct = None   # a number from 1 to 4

Build `V5`, `rows5 = top_k(V5 @ q, 4)` and `distinct5` = the number of different chunks (`r % 91`) among `rows5`.

In [ ]:
# Your code here (Task 5b). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 5b (do not edit)
assert my_prediction_distinct is not None, "write your prediction first"
assert V5.shape == (455, 384), "V5 must hold 5 copies of every row"
assert _fp([int(r) for r in rows5]) == "f65d5e725ddb1190", f"rows5 = {list(map(int, rows5))}: equal scores must go to the smaller row number"
assert _fp(int(distinct5)) == "6b86b273ff34fce1", f"distinct5 = {distinct5} is not right yet"
print(f"Task 5b OK. Predicted {my_prediction_distinct} different chunks; got {distinct5}: rows {list(map(int, rows5))}")

## Task 6: a metadata filter, before or after the search

You only want chunks from page 25 onwards: `keep = lambda m: m["page"] >= 25`.

Write two functions that return a list of chunk ids (at most k):
- `search_prefilter(V, meta, q, k, keep)`: drop every chunk that fails `keep` **before** choosing the top-k (hint: give it a score of `-np.inf`), and never return a dropped chunk;
- `search_postfilter(V, meta, q, k, keep, fetch_k=20)`: take the top-`fetch_k` **first**, then drop what fails `keep`, then keep the first k. This is how LangChain's FAISS store applies a `filter` (lesson D.4).

Queries: `X` = the embeddings of `[Q_D002, Q_D008]` followed by the first 200 characters (`.strip()[:200]`) of each of the 33 pages: shape `(35, 384)`, normalised.

**Predict first.** Out of the 35 queries, for how many will the post-filter return **fewer than 4** chunks?

In [ ]:
my_prediction_post_short = None   # a number from 0 to 35

Build `keep`, `X`, the two functions, and the counts `n_pre_short` and `n_post_short` (queries that got fewer than 4 results).

In [ ]:
# Your code here (Task 6). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 6 (do not edit)
assert my_prediction_post_short is not None, "write your prediction first"
assert X.shape == (35, 384), f"X must be (35, 384), got {X.shape}"
assert all(keep(meta[i]) for x in X for i in search_prefilter(V, meta, x, 4, keep)), "the pre-filter returned a chunk that fails the filter"
assert _fp(search_prefilter(V, meta, X[0], 4, keep)) == "0a3fb4940ee86b6b", "pre-filter ids for D-002 are not right yet (return a list of int)"
assert _fp(int(n_pre_short)) == "5feceb66ffc86f38", f"n_pre_short = {n_pre_short} is not right yet"
assert _fp(int(n_post_short)) == "19581e27de7ced00", f"n_post_short = {n_post_short} is not right yet (fetch_k=20, then filter, then first k)"
print(f"Task 6 OK. Predicted {my_prediction_post_short}; the post-filter came back short for {n_post_short}/35 queries, the pre-filter for {n_pre_short}/35")

## Task 7: the distance is a hidden default

A store picks a distance for you: LangChain's FAISS store uses L2 (Euclidean) distance unless told otherwise (lesson D.5). Does that change *which* chunks come back?

1. On `V` (unit vectors): for each of the 35 queries in `X`, compare the top-4 by **dot product**, by **cosine** and by **smallest L2 distance** (rank by `-distance`). Count `n_dot_eq_cos` and `n_l2_eq_cos` (queries where the top-4 lists are identical).
2. **Toy illustration:** give each chunk vector a different length, as an embedder without normalisation could: `r = np.random.default_rng(1).uniform(0.5, 1.5, size=91).astype(np.float32)` and `U = V * r[:, None]`. Count `m_dot_eq_cos` and `m_l2_eq_cos` on `U`.

**Predict first.** On the toy `U`, for how many of the 35 queries will dot product and cosine give the same top-4?

In [ ]:
my_prediction_toy_dot_eq_cos = None   # a number from 0 to 35

In [ ]:
# Your code here (Task 7). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 7 (do not edit)
assert my_prediction_toy_dot_eq_cos is not None, "write your prediction first"
assert U.shape == V.shape and not np.allclose(np.linalg.norm(U, axis=1), 1.0), "U must be V with rows of different lengths"
assert _fp(int(n_dot_eq_cos)) == "9f14025af0065b30" and _fp(int(n_l2_eq_cos)) == "9f14025af0065b30", \
    f"on unit vectors you got dot={n_dot_eq_cos}, L2={n_l2_eq_cos}: check that cosine divides by both lengths and L2 ranks smallest first"
assert _fp(int(m_dot_eq_cos)) == "5feceb66ffc86f38", f"m_dot_eq_cos = {m_dot_eq_cos} is not right yet"
assert _fp(int(m_l2_eq_cos)) == "6b86b273ff34fce1", f"m_l2_eq_cos = {m_l2_eq_cos} is not right yet"
print(f"Task 7 OK. Unit vectors: dot = cosine = L2 ranking for {n_dot_eq_cos}/35. Toy U: dot = cosine for {m_dot_eq_cos}/35 (you predicted {my_prediction_toy_dot_eq_cos}), L2 = cosine for {m_l2_eq_cos}/35")

## Task 8: when is exact search enough? (CR-08)

Write `time_search(N, d=384, k=4)`: make `N` random unit vectors of dimension `d` (`float32`, seeded), one random unit query, and return the **median** seconds of `top_k(A @ q, k)` over enough repeats (at least 7; more for small N). Do one warm-up call first.

**Predict first** (before you time anything):
1. our real index (91 chunks): how many **milliseconds** per query?
2. if **k doubles** from 4 to 8 at N = 100,000, the time is multiplied by about ____ ?
3. if **N doubles** from 100,000 to 200,000, the time is multiplied by about ____ ?

In [ ]:
my_prediction_ms_91 = None
my_prediction_ratio_k = None
my_prediction_ratio_N = None

Then measure and store (in seconds): `t_91 = time_search(91)`, `t_10k = time_search(10_000)`, `t_100k = time_search(100_000)`, `t_100k_k8 = time_search(100_000, k=8)`, `t_200k = time_search(200_000)`. Memory check: an index of N vectors of dimension d in float32 takes N × d × 4 bytes; compute `mb_200k` for N = 200,000, d = 384.

Timings depend on your machine. If the CHECK fails on a timing bound, close other programs and rerun this cell once.

In [ ]:
# Your code here (Task 8). Blank-file first: write it yourself; no copying from p5_measure.py.

In [ ]:
# CHECK Task 8 (do not edit)
assert None not in (my_prediction_ms_91, my_prediction_ratio_k, my_prediction_ratio_N), "write all three predictions first"
assert abs(mb_200k - 307.2) < 0.5, f"mb_200k = {mb_200k}: N x d x 4 bytes, in MB (1 MB = 1,000,000 bytes)"
assert t_91 < t_10k < t_100k < t_200k, "time should grow with N (rerun once if the machine was busy)"
ratio_k, ratio_N = t_100k_k8 / t_100k, t_200k / t_100k
assert ratio_k < 1.5, f"k 4 -> 8 changed the time x{ratio_k:.2f}; exact search time should barely depend on k (rerun once if busy)"
assert 1.3 < ratio_N < 3.5, f"N x2 changed the time x{ratio_N:.2f}; expected roughly x2 (rerun once if busy)"
print(f"Task 8 OK. 91 chunks: {t_91*1e3:.3f} ms (you predicted {my_prediction_ms_91}). k x2: x{ratio_k:.2f} (you predicted {my_prediction_ratio_k}). N x2: x{ratio_N:.2f} (you predicted {my_prediction_ratio_N}).")
print(f"10k: {t_10k*1e3:.3f} ms, 100k: {t_100k*1e3:.3f} ms, 200k: {t_200k*1e3:.3f} ms; 200k x 384 float32 = {mb_200k:.1f} MB")

## Task 9: the evidence sentence for the 02 card

In 2-3 sentences, using **your** numbers from Task 8: why does v0 not need an ANN index (HNSW or IVF)? Say what would have to change before it does.

*Template:* "Our index has ____ vectors and exact search takes ____ ms per query, so ____. ANN would only pay off when ____, and it would cost ____."

In [ ]:
my_answer_why_no_ann = ""   # write your 2-3 sentences inside the quotes

In [ ]:
# CHECK Task 9 (do not edit)
assert len(my_answer_why_no_ann.strip()) >= 80, "write 2-3 full sentences"
assert any(ch.isdigit() for ch in my_answer_why_no_ann), "use at least one of your measured numbers"
print("Task 9 recorded. The tutor grades it with the Transfer rubric (05 §4.2).")

## After the lab

- Log in the session record: which tasks needed a hint (and which level), your predictions vs the measured values, and any failing CHECK you fixed yourself.
- The tutor grades `coding_independence_0_3` on Tasks 3, 4 and 6, and `debugging_0_3` on any failing CHECK you diagnosed and fixed.
- Then go back to the lesson, Part J.